# AI 기본법 선택적 계층 청킹 Qdrant 테스트

중복되는 조·항·호를 모두 벡터화하지 않고 약 200개 수준으로 줄입니다.

- article: 모든 조문의 직접 조문내용
- paragraph: 직접 항내용이 있는 항
- item: 항내용이 없고 호 목록으로만 구성된 항의 호
- context_content: 답변에 사용할 하위 전체 내용

제2조 정의는 호 단위로 검색하고, 일반 조문은 조·항 단위로 검색합니다.


In [1]:
from pathlib import Path
import sys
import re
import uuid
import xml.etree.ElementTree as ET
from collections import Counter

from dotenv import load_dotenv
from qdrant_client.models import Distance, PointStruct, VectorParams

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "common" / "ai_model.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 폴더에서 실행하세요.")

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client

XML_PATH = PROJECT_ROOT / "data" / "ai_basic_law" / "ai_basic_law.xml"
if not XML_PATH.exists():
    raise FileNotFoundError(f"XML 파일이 없습니다: {XML_PATH}")

root = ET.parse(XML_PATH).getroot()
for element in root.iter():
    element.tag = element.tag.rsplit("}", 1)[-1]

print("XML:", XML_PATH)
print("루트:", root.tag)


XML: /home/aiuser/rag_mini_pjt_template/data/ai_basic_law/ai_basic_law.xml
루트: 법령


def normalize(text):
    return ' '.join((text or '').split())

def direct_text(node, tag):
    return normalize(node.findtext(f'./{tag}', ''))

def descendant_content(node, tags):
    parts = []
    for element in node.iter():
        if element.tag not in tags:
            continue
        text = normalize(''.join(element.itertext()))
        if text and text not in parts:
            parts.append(text)
    return '\n'.join(parts)

def stable_id(*parts):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, '/'.join(map(str, parts))))

chunks = []
SECTION_ONLY = re.compile(r'^제\s*\d+\s*[편장절관]\s*')
law_name = normalize(root.findtext('./기본정보/법령명_한글', ''))
effective_date = normalize(root.findtext('./기본정보/시행일자', ''))

def add_chunk(chunk_type, article, title, content, context_content,
              paragraph='', item='', parent_id=None):
    if not content:
        return
    chunk_id = stable_id(law_name, effective_date, article, chunk_type, paragraph, item)
    chunks.append({
        'id': chunk_id,
        'node_id': chunk_id,
        'parent_id': parent_id,
        'chunk_type': chunk_type,
        'article': article,
        'paragraph': paragraph,
        'item': item,
        'title': title,
        'content': content,
        'context_content': context_content or content,
        'law_name': law_name,
        'effective_date': effective_date,
        'source': XML_PATH.name,
    })

for unit in root.findall('.//조문단위'):
    number = normalize(unit.findtext('조문번호', ''))
    branch = normalize(unit.findtext('조문가지번호', ''))
    article = f'제{number}조' + (f'의{branch}' if branch and branch != '0' else '')
    title = normalize(unit.findtext('조문제목', ''))
    article_direct = direct_text(unit, '조문내용')

    if not title and SECTION_ONLY.match(article_direct):
        continue

    article_id = stable_id(law_name, effective_date, article, 'article')
    article_context = descendant_content(
        unit, {'조문내용', '항내용', '호내용', '목내용'}
    )
    add_chunk(
        'article', article, title,
        article_direct, article_context,
    )

    for paragraph_index, paragraph_node in enumerate(unit.findall('./항'), start=1):
        paragraph_number = normalize(paragraph_node.findtext('항번호', ''))
        paragraph = paragraph_number or f'{paragraph_index}항'
        paragraph_id = stable_id(article_id, 'paragraph', paragraph_index)
        paragraph_direct = direct_text(paragraph_node, '항내용')
        paragraph_context = descendant_content(
            paragraph_node, {'항내용', '호내용', '목내용'}
        )

        add_chunk(
            'paragraph', article, title,
            paragraph_direct, paragraph_context,
            paragraph=paragraph,
            parent_id=article_id,
        )

        if not paragraph_direct:
            for item_index, item_node in enumerate(paragraph_node.findall('./호'), start=1):
                item_number = normalize(item_node.findtext('호번호', ''))
                item = item_number or f'{item_index}호'
                item_direct = direct_text(item_node, '호내용')
                item_context = descendant_content(
                    item_node, {'호내용', '목내용'}
                )
                add_chunk(
                    'item', article, title,
                    item_direct, item_context,
                    paragraph=paragraph,
                    item=item,
                    parent_id=paragraph_id,
                )

assert chunks, '생성된 청크가 없습니다.'
assert len(chunks) == len({chunk['id'] for chunk in chunks}), '청크 ID가 중복되었습니다.'

print(f'선택적 계층 청크 수: {len(chunks)}개')
print('유형별:', dict(Counter(chunk['chunk_type'] for chunk in chunks)))


In [2]:
article_2_chunks = [
    chunk for chunk in chunks
    if chunk['article'] == '제2조'
]

for chunk in article_2_chunks:
    print(
        f"[{chunk['chunk_type']}]",
        chunk['article'],
        chunk['paragraph'],
        chunk['item'],
        chunk['title'],
    )
    print('[검색용]', chunk['content'][:500])
    print('[답변용]', chunk['context_content'][:1000])
    print()

definition_chunks = [
    chunk for chunk in article_2_chunks
    if chunk['chunk_type'] == 'item'
    and '"고영향 인공지능"이란' in chunk['content']
]
assert len(definition_chunks) == 1, '고영향 인공지능 정의 청크를 하나 찾지 못했습니다.'
print('제2조제4호 검색 청크 확인 완료')


전체 청크: 403
유형별: {'article': 46, 'paragraph': 166, 'item': 191}


embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = 'ai_basic_law_selective_hierarchy_v1'

def embedding_text(chunk):
    location = ' '.join(
        value for value in [
            chunk['law_name'],
            chunk['article'],
            chunk['paragraph'],
            chunk['item'],
            chunk['title'],
        ] if value
    )
    return f"{location}\n{chunk['content']}"

vectors = await embedding_client.aembed_documents(
    [embedding_text(chunk) for chunk in chunks]
)

if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=len(vectors[0]),
        distance=Distance.COSINE,
    ),
)

BATCH_SIZE = 64
for start in range(0, len(chunks), BATCH_SIZE):
    chunk_batch = chunks[start:start + BATCH_SIZE]
    vector_batch = vectors[start:start + BATCH_SIZE]
    points = [
        PointStruct(id=chunk['id'], vector=vector, payload=chunk)
        for chunk, vector in zip(chunk_batch, vector_batch)
    ]
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True,
    )

print(f'{len(chunks)}개 청크 저장 완료: {COLLECTION_NAME}')


In [3]:
query = '고영향 인공지능이란 무엇인가요?'
query_vector = await embedding_client.aembed_query(query)
result = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=10,
)

for rank, point in enumerate(result.points, start=1):
    payload = point.payload
    print(
        rank,
        f"[{payload['chunk_type']}]",
        payload['article'],
        payload['paragraph'],
        payload['item'],
        payload['title'],
        f"score={point.score:.4f}",
    )
    print('[검색용]', payload['content'])
    print('[답변용]', payload.get('context_content', payload['content'])[:1500])
    print()


[article] 제2조   정의
제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2026.1.20>
1. "인공지능"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다.
2. "인공지능시스템"이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다.
3. "인공지능기술"이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다.
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.
가. 「에너지법」 제2조제1호에 따른 에너지의 공급
나. 「먹는물관리법」 제3조제1호에 따른 먹는물의 생산 공정
다. 「보건의료기본법」 제3조제1호에 따른 보건의료의 제공 및 이용체계의 구축ㆍ운영
라. 「의료기기법」 제2조제1항에 따른 의료기기 및 「디지털의료제품법」 제2조제2호에 따른 디지털의료기기의 개발 및 이용
마. 「원자력시설 등의 방호 및 방사능 방재 대책법」 제2조제1항제1호에 따른 핵물질과 같은 항 제2호에 따른 원자력시설의 안전한 관리 및 운영
바. 범죄 수사나 체포 업무를 위한 생체인식정보(얼굴ㆍ지문ㆍ홍채 및 손바닥 정맥 등 개인을 식별할 수 있는 신체적ㆍ생리적ㆍ행동적 특징에 관한 개인정보를 말한다)의 분석ㆍ활용
사. 채용, 대출 심사 등 개인의 권리ㆍ의무 관계에 중대한 영향을 미치는 판단 또는 평가
아. 「교통안전법」 제2조제1호부터 제3호까지에 따른 교통수단, 교통시설, 교통체계의 주요한 작동 및 운영
자. 공공서비스 제공에 필요한 자격 확인 및 결정 또는 비용징수 등 국민에게 영향을 미치는 국가, 지방자치단체, 「공공기관의 운영에 관한 법률」 제4조에 따른 공공기관 등(이하 "국가기관등"이라 한

## 3. 임베딩과 Qdrant 저장

임베딩 입력에는 조·항·호와 제목을 포함합니다. Payload에는 구조 metadata와 원문 내용을 저장합니다.

새 컬렉션: `ai_basic_law_article_paragraph_item_v1`


In [4]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = "ai_basic_law_article_paragraph_item_v1"

def embedding_text(chunk):
    location = " ".join(
        part
        for part in [
            chunk["law_name"],
            chunk["article"],
            chunk["paragraph"],
            chunk["item"],
            chunk["title"],
        ]
        if part
    )
    return f"{location}\n{chunk['content']}"

texts = [embedding_text(chunk) for chunk in chunks]
vectors = await embedding_client.aembed_documents(texts)

if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=len(vectors[0]),
        distance=Distance.COSINE,
    ),
)

BATCH_SIZE = 64
for start in range(0, len(chunks), BATCH_SIZE):
    chunk_batch = chunks[start:start + BATCH_SIZE]
    vector_batch = vectors[start:start + BATCH_SIZE]
    points = [
        PointStruct(
            id=chunk["id"],
            vector=vector,
            payload=chunk,
        )
        for chunk, vector in zip(chunk_batch, vector_batch)
    ]
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True,
    )

print(f"{len(chunks)}개 청크 저장 완료: {COLLECTION_NAME}")


403개 청크 저장 완료: ai_basic_law_article_paragraph_item_v1


## 4. 검색 확인

정상이라면 제2조의 `item` 청크가 상위 결과에 포함되어야 합니다.


In [5]:
query = "고영향 인공지능이란 무엇인가요?"
query_vector = await embedding_client.aembed_query(query)

result = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=10,
)

for rank, point in enumerate(result.points, start=1):
    payload = point.payload
    print(
        rank,
        f"[{payload['chunk_type']}]",
        payload["article"],
        payload["paragraph"],
        payload["item"],
        payload["title"],
        f"score={point.score:.4f}",
    )
    print(payload["content"][:1000])
    print()


1 [item] 제34조 ① 6. 고영향 인공지능과 관련한 사업자의 책무 score=0.6065
6. 그 밖에 고영향 인공지능의 안전성ㆍ신뢰성 확보를 위하여 위원회에서 심의ㆍ의결된 사항

2 [paragraph] 제35조 ①  고영향 인공지능 영향평가 score=0.6062
① 인공지능사업자가 고영향 인공지능을 이용한 제품 또는 서비스를 제공하는 경우 사전에 사람의 기본권에 미치는 영향을 평가(이하 "영향평가"라 한다)하기 위하여 노력하여야 한다. 이 경우 영향평가에는 고영향 인공지능을 이용한 제품 또는 서비스의 성격을 고려하여 인공지능취약계층의 특성이 반영될 수 있도록 하여야 한다. <개정 2026.1.20>

3 [item] 제8조 ① 10. 위원회의 기능 score=0.6034
10. 고영향 인공지능 규율에 관한 사항

4 [paragraph] 제33조 ①  고영향 인공지능의 확인 score=0.5889
① 인공지능사업자는 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 그 인공지능이 고영향 인공지능에 해당하는지에 대하여 사전에 검토하여야 하며, 필요한 경우 과학기술정보통신부장관에게 고영향 인공지능에 해당하는지 여부의 확인을 요청할 수 있다.

5 [paragraph] 제34조 ①  고영향 인공지능과 관련한 사업자의 책무 score=0.5853
① 인공지능사업자는 고영향 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 고영향 인공지능의 안전성ㆍ신뢰성을 확보하기 위하여 다음 각 호의 내용을 포함하는 조치를 대통령령으로 정하는 바에 따라 이행하여야 한다.
1. 위험관리방안의 수립ㆍ운영
2. 기술적으로 가능한 범위에서의 인공지능이 도출한 최종결과, 인공지능의 최종결과 도출에 활용된 주요 기준, 인공지능의 개발ㆍ활용에 사용된 학습용데이터의 개요 등에 대한 설명 방안의 수립ㆍ시행
3. 이용자 보호 방안의 수립ㆍ운영
4. 고영향 인공지능에 대한 사람의 관리ㆍ감독
5. 안전성ㆍ신뢰성 확보를 위한 조치의 내용을 확인할 수 있는 문서의 작성과 보관
6. 그

## 다음 단계

Advanced RAG 평가 노트북의 컬렉션 이름을 다음으로 변경합니다.

```python
COLLECTION_NAME = "ai_basic_law_article_paragraph_item_v1"
```

Retrieval 평가는 조문 전체·항·호가 동시에 검색되어도 같은 정답 근거를 중복 계산하지 않도록 조문 기준으로 정규화합니다.
